In [ ]:
import asyncio
import aiohttp
import time
import hmac
import hashlib
import json
import numpy as np
import requests
import threading
import sys
import warnings
from collections import deque
from pybit.unified_trading import WebSocket
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)
warnings.simplefilter('ignore', UserWarning)

# ==========================================
# 🏛️ TITAN V6.0: SINGLE-CORE PROP FIRM ENGINE
# ==========================================
print("--- 🏛️ TITAN V6: INSTITUTIONAL MAKER EDITION ---")
print("  🎯 Target: ADAUSDT / XRPUSDT")
print("  🛡️ Risk: 3% Size | 3.5 Stop | 0.0 Exit (Prop Firm Safe)")

# 1. YOUR KEYS (Currently set to BYBIT TESTNET)
API_KEY = "3vGBUlSFvnqvtNzSZQ"
API_SECRET = "A2OMx50gzwlPBg6w3NIxmJGdl0fy0LRL4E01"
BASE_URL = "https://api-demo.bybit.com"

# 🛡️ ICEBERG CHUNKER: Max Notional Size Per Sub-Order to bypass Exchange Limits
MAX_ORDER_NOTIONAL_USD = 50000 

# 2. THE V6 OPTIMIZED CONFIGURATION
STRATEGY = {
    "coin_a": "ADAUSDT", "coin_b": "XRPUSDT",
    "timeframe": 60, "window": 168,
    "z_entry": 2.5, "z_exit": 0.0, "z_stop": 3.5,
    "pct_size": 0.03, "leverage": 10,
    "order_type": "LimitMaker", # 💰 Post-Only Maker Orders to crush fees
    "prices_a": deque(maxlen=168), "prices_b": deque(maxlen=168),
    "status": "NEUTRAL", "cooldown_until": 0.0, "current_interval": 0
}

# Global Memory for WS
latest_prices = {"ADAUSDT": None, "XRPUSDT": None}
time_offset = 0.0  

# ==========================================
# ⚡ BACKGROUND ASYNC DAEMON
# ==========================================
background_loop = asyncio.new_event_loop()
def start_background_loop(loop):
    asyncio.set_event_loop(loop)
    loop.run_forever()
threading.Thread(target=start_background_loop, args=(background_loop,), daemon=True).start()

# ==========================================
# ⏳ CLOCK SYNC & PRE-FILL
# ==========================================
print("\n⏳ SYNCING CLOCK & DOWNLOADING 168-HOUR MACRO DATA...")
try:
    res = requests.get(BASE_URL + "/v5/market/time").json()
    time_offset = (float(res['result']['timeNano']) / 1e9) - time.time()
except Exception: pass

def prefill_data(coin, window):
    prices = deque(maxlen=window)
    try:
        res = requests.get(f"{BASE_URL}/v5/market/kline?category=linear&symbol={coin}&interval=60&limit={window}").json()
        for candle in reversed(res['result']['list']): prices.append(float(candle[4]))
    except Exception as e: print(f"⚠️ Pre-fill failed for {coin}: {e}")
    return prices

STRATEGY["prices_a"] = prefill_data(STRATEGY["coin_a"], STRATEGY["window"])
STRATEGY["prices_b"] = prefill_data(STRATEGY["coin_b"], STRATEGY["window"])
STRATEGY["current_interval"] = int(time.time() + time_offset) // (60 * STRATEGY["timeframe"])
print(f"✅ Data Loaded: {STRATEGY['coin_a']} / {STRATEGY['coin_b']}")

# ==========================================
# ⚡ ASYNC EXECUTION & ICEBERG CHUNKER
# ==========================================
def generate_signature(payload, time_stamp):
    param_str = str(time_stamp) + API_KEY + "5000" + payload
    hash_obj = hmac.new(bytes(API_SECRET, "utf-8"), param_str.encode("utf-8"), hashlib.sha256)
    return hash_obj.hexdigest()

async def get_wallet_balance(session):
    time_stamp = str(int((time.time() + time_offset) * 1000))
    payload = "accountType=UNIFIED&coin=USDT"
    sign = generate_signature(payload, time_stamp)
    headers = {'X-BAPI-API-KEY': API_KEY, 'X-BAPI-SIGN': sign, 'X-BAPI-SIGN-TYPE': '2', 'X-BAPI-TIMESTAMP': time_stamp, 'X-BAPI-RECV-WINDOW': '5000'}
    try:
        async with session.get(BASE_URL + "/v5/account/wallet-balance?" + payload, headers=headers) as response:
            res = await response.json()
            return float(res['result']['list'][0]['totalEquity'])
    except Exception: return 0.0

async def async_place_order(session, symbol, side, qty, order_type="LimitMaker", price=None):
    time_stamp = str(int((time.time() + time_offset) * 1000))
    if order_type == "LimitMaker" and price is not None:
        payload_dict = {"category": "linear", "symbol": symbol, "side": side, "orderType": "Limit", "qty": str(qty), "price": str(price), "timeInForce": "PostOnly"}
    else:
        payload_dict = {"category": "linear", "symbol": symbol, "side": side, "orderType": "Market", "qty": str(qty), "timeInForce": "IOC"}
    
    payload = json.dumps(payload_dict)
    sign = generate_signature(payload, time_stamp)
    headers = {'X-BAPI-API-KEY': API_KEY, 'X-BAPI-SIGN': sign, 'X-BAPI-SIGN-TYPE': '2', 'X-BAPI-TIMESTAMP': time_stamp, 'X-BAPI-RECV-WINDOW': '5000', 'Content-Type': 'application/json'}
    try:
        async with session.post(BASE_URL + "/v5/order/create", headers=headers, data=payload) as response:
            result = await response.json()
            if result['retCode'] == 0: print(f"✅ [{symbol}] {side} {qty} Fill Success.")
            else: print(f"❌ [{symbol}] {side} Fill Failed: {result['retMsg']}")
    except Exception: pass

async def async_close_position(session, symbol, order_type="Market", limit_price=None):
    time_stamp = str(int((time.time() + time_offset) * 1000))
    payload = f"category=linear&symbol={symbol}"
    sign = generate_signature(payload, time_stamp)
    headers = {'X-BAPI-API-KEY': API_KEY, 'X-BAPI-SIGN': sign, 'X-BAPI-SIGN-TYPE': '2', 'X-BAPI-TIMESTAMP': time_stamp, 'X-BAPI-RECV-WINDOW': '5000'}
    try:
        async with session.get(BASE_URL + "/v5/position/list?" + payload, headers=headers) as response:
            res = await response.json()
            if res.get('result', {}).get('list'):
                pos = res['result']['list'][0]
                size = float(pos['size'])
                if size > 0:
                    close_side = "Sell" if pos['side'] == "Buy" else "Buy"
                    current_price = latest_prices.get(symbol, 1.0)
                    max_qty_per_chunk = max(1, int(MAX_ORDER_NOTIONAL_USD / current_price))
                    
                    remaining_size = int(size)
                    print(f"🔄 Executing Close: {close_side} {remaining_size} {symbol}...")
                    
                    while remaining_size > 0:
                        chunk_qty = min(remaining_size, max_qty_per_chunk)
                        await async_place_order(session, symbol, close_side, chunk_qty, order_type, limit_price)
                        remaining_size -= chunk_qty
                        if remaining_size > 0: await asyncio.sleep(0.5) 
    except Exception: pass

def close_all_positions(reason, force_market=False):
    print(f"\n🛑 CLOSING POSITIONS ({reason})...")
    o_type = "Market" if force_market else STRATEGY["order_type"]
    
    async def execute_close():
        async with aiohttp.ClientSession() as session:
            await asyncio.gather(
                async_close_position(session, STRATEGY["coin_a"], o_type, latest_prices[STRATEGY["coin_a"]]),
                async_close_position(session, STRATEGY["coin_b"], o_type, latest_prices[STRATEGY["coin_b"]])
            )
            
    asyncio.run_coroutine_threadsafe(execute_close(), background_loop)
    STRATEGY["status"] = "NEUTRAL"
    STRATEGY["cooldown_until"] = time.time() + 3600  
    print(f"⏳ Cooldown activated for 1 HOUR. Letting the spread reset.")

# ==========================================
# 🧠 DYNAMIC SIZING & ARIMA VETO
# ==========================================
def calculate_dynamic_sizes(equity):
    arr_a, arr_b = np.array(STRATEGY["prices_a"]), np.array(STRATEGY["prices_b"])
    ret_a, ret_b = np.diff(np.log(arr_a)), np.diff(np.log(arr_b))
    vol_a, vol_b = np.std(ret_a), np.std(ret_b)
    
    if vol_a < 1e-6 or vol_b < 1e-6 or (vol_a + vol_b) == 0: weight_a = weight_b = 0.5
    else: weight_a, weight_b = vol_b / (vol_a + vol_b), vol_a / (vol_a + vol_b)
    
    total_notional = equity * STRATEGY["pct_size"] * STRATEGY["leverage"]
    alloc_a = min(total_notional * weight_a, MAX_ORDER_NOTIONAL_USD)
    alloc_b = min(total_notional * weight_b, MAX_ORDER_NOTIONAL_USD)
    
    qty_a = max(1, int(alloc_a / latest_prices[STRATEGY["coin_a"]]))
    qty_b = max(1, int(alloc_b / latest_prices[STRATEGY["coin_b"]]))
    return qty_a, qty_b

def arima_veto_check(ratio_array, direction):
    try:
        model = ARIMA(ratio_array, order=(1, 0, 1))
        fitted = model.fit()
        forecast = fitted.forecast(steps=3) 
        if direction == "SHORT_SPREAD" and forecast[-1] > ratio_array[-1]: return False 
        if direction == "LONG_SPREAD" and forecast[-1] < ratio_array[-1]: return False
        return True
    except Exception: return True

async def execute_dynamic_pairs_trade(side_a, side_b):
    async with aiohttp.ClientSession() as session:
        equity = await get_wallet_balance(session)
        if equity == 0: return
        qty_a, qty_b = calculate_dynamic_sizes(equity)
        
        print(f"🚀 FIRING: {side_a} {qty_a} {STRATEGY['coin_a']} / {side_b} {qty_b} {STRATEGY['coin_b']}")
        await asyncio.gather(
            async_place_order(session, STRATEGY["coin_a"], side_a, qty_a, STRATEGY["order_type"], latest_prices[STRATEGY["coin_a"]]),
            async_place_order(session, STRATEGY["coin_b"], side_b, qty_b, STRATEGY["order_type"], latest_prices[STRATEGY["coin_b"]])
        )

# ==========================================
# 🧠 Z-SCORE MACRO TRIGGER ENGINE
# ==========================================
def process_strategies():
    if time.time() < STRATEGY["cooldown_until"]: return
    if len(STRATEGY["prices_a"]) < STRATEGY["window"] or len(STRATEGY["prices_b"]) < STRATEGY["window"]: return

    ratio = np.array(STRATEGY["prices_a"]) / np.array(STRATEGY["prices_b"])
    mean, std = np.mean(ratio), np.std(ratio)
    if std < 0.0001: return  
    
    z_score = (ratio[-1] - mean) / std
    
    # 1. CIRCUIT BREAKER
    if abs(z_score) >= STRATEGY["z_stop"]:
        if STRATEGY["status"] == "ACTIVE TRADE":
            print(f"\n⛔ DANGER: Z-Score hit {z_score:.2f}. Triggering STOP LOSS!")
            close_all_positions(reason="STOP LOSS", force_market=True)
        return 

    # 2. ENTRY LOGIC
    if STRATEGY["z_entry"] <= z_score < STRATEGY["z_stop"] and STRATEGY["status"] == "NEUTRAL":
        print(f"\n📈 SIGNAL [Z: {z_score:.2f}]: Short {STRATEGY['coin_a']}, Long {STRATEGY['coin_b']}.")
        if arima_veto_check(ratio, "SHORT_SPREAD"):
            STRATEGY["status"] = "ACTIVE TRADE"
            asyncio.run_coroutine_threadsafe(execute_dynamic_pairs_trade("Sell", "Buy"), background_loop)
            
    elif -STRATEGY["z_stop"] < z_score <= -STRATEGY["z_entry"] and STRATEGY["status"] == "NEUTRAL":
        print(f"\n📉 SIGNAL [Z: {z_score:.2f}]: Long {STRATEGY['coin_a']}, Short {STRATEGY['coin_b']}.")
        if arima_veto_check(ratio, "LONG_SPREAD"):
            STRATEGY["status"] = "ACTIVE TRADE"
            asyncio.run_coroutine_threadsafe(execute_dynamic_pairs_trade("Buy", "Sell"), background_loop)
            
    # 3. PROFIT TARGET (FULL REVERSION TO 0.0)
    elif STRATEGY["status"] == "ACTIVE TRADE" and abs(z_score) <= STRATEGY["z_exit"]:
        print(f"\n🎯 TARGET REACHED [Z: {z_score:.2f}]. Closing positions.")
        close_all_positions(reason="PROFIT", force_market=False)

# ==========================================
# 📡 TRUE 1-HOUR AGGREGATED WEBSOCKET
# ==========================================
def handle_message(message):
    try:
        data = message.get('data', {})
        data = data[0] if isinstance(data, list) else data
        symbol = data.get('symbol')
        
        price_str = data.get('markPrice')
        if not price_str: price_str = data.get('lastPrice')
        
        if not price_str: return
        latest_prices[symbol] = float(price_str)
        
        live_interval = int(time.time() + time_offset) // 3600 
        
        if latest_prices[STRATEGY["coin_a"]] is None or latest_prices[STRATEGY["coin_b"]] is None: return
            
        if live_interval > STRATEGY["current_interval"]:
            STRATEGY["prices_a"].append(latest_prices[STRATEGY["coin_a"]])
            STRATEGY["prices_b"].append(latest_prices[STRATEGY["coin_b"]])
            STRATEGY["current_interval"] = live_interval
        else:
            if len(STRATEGY["prices_a"]) > 0: STRATEGY["prices_a"][-1] = latest_prices[STRATEGY["coin_a"]]
            if len(STRATEGY["prices_b"]) > 0: STRATEGY["prices_b"][-1] = latest_prices[STRATEGY["coin_b"]]
            
        process_strategies()
    except Exception: pass 

print("📡 CONNECTING TO WEBSOCKET STREAM...")
ws = WebSocket(testnet=True, channel_type="linear")
for coin in latest_prices.keys():
    ws.ticker_stream(symbol=coin, callback=handle_message)
print("✅ STREAM CONNECTED.")

# ==========================================
# 🏁 MAIN KEEPALIVE LOOP
# ==========================================
while True:
    try:
        time.sleep(2)
        print(f"\r⏳ Heartbeat | {STRATEGY['coin_a']} / {STRATEGY['coin_b']} | Status: {STRATEGY['status']}    ", end="")
    except KeyboardInterrupt:
        print("\n🛑 Shutting down TITAN V6...")
        sys.exit()